In [17]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017/")  
db = client["yeogi"]                                
collection = db["accommodations"]                   


service = Service(ChromeDriverManager().install())
options = Options()
options.add_argument('--disable-gpu')
options.add_argument('--disable-dev-shm-usage')
options.add_argument('--start-maximized')
options.add_argument('--user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/139.0.0.0 Safari/537.36')
options.add_argument('--lang=ko_KR')

driver = webdriver.Chrome(service=service, options=options)
url = "https://www.yeogi.com/"
driver.get(url)
time.sleep(2)

wait = WebDriverWait(driver, 5)


search_button = driver.find_element(By.CSS_SELECTOR, ".css-59ixa7")
search_button.click()


gyeongju_item = wait.until(
    EC.element_to_be_clickable((By.CSS_SELECTOR, 'li[data-keyword="경주"]'))
)
gyeongju_item.click()
time.sleep(1)

search_btn = wait.until(
    EC.element_to_be_clickable((By.CSS_SELECTOR, 'button.gc-box-button.css-1h4agld'))
)
search_btn.click()
time.sleep(2)

data_list = []
scroll_pause = 2

for page in range(1, 4):  
    for _ in range(10):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(scroll_pause)

    cards = driver.find_elements(By.CSS_SELECTOR, "div.gc-thumbnail-type-seller-card-wrapper")

    for card in cards:
        try:
            category = card.find_element(By.CSS_SELECTOR, "ul.css-bl7zf6").text.strip()
        except:
            category = None

        try:
            name = card.find_element(By.CSS_SELECTOR, "h3.gc-thumbnail-type-seller-card-title").text.strip()
        except:
            name = None

        try:
            location = card.find_element(By.CSS_SELECTOR, "span.css-1rzfout").text.strip()
        except:
            location = None
        try:
            rating = card.find_element(By.XPATH, ".//span[normalize-space(text()) and contains(text(),'.')]").text.strip()
        except:
            rating = None

       
        try:
            reviews_raw = card.find_element(By.CSS_SELECTOR, "span.css-oj6onp").text
            reviews = reviews_raw.replace("\n", " ").strip()   # "8,531 명 평가"
        except:
            reviews = None

 
        try:
            price = card.find_element(By.CSS_SELECTOR, "span.css-5r5920").text.strip()
        except:
            price = None

        data_list.append({
            "page": page,
            "category": category,
            "name": name,
            "location": location,
            "rating": rating,
            "reviews": reviews,
            "price": price
        })


    if page < 3:  
        next_btn = wait.until(
            EC.element_to_be_clickable((By.XPATH, f'//button[text()="{page+1}"]'))
        )
        driver.execute_script("arguments[0].click();", next_btn)
        time.sleep(3)  

for d in data_list:
    print(d)

driver.quit()

if data_list:
    collection.insert_many(data_list)
    print(f"MongoDB 저장 완료: {len(data_list)} 건")
else:
    print("저장할 데이터가 없습니다.")


1 페이지 스크롤 완료, 데이터 수집 시작
1 페이지 데이터 수집 완료 → 총 20건
2 페이지 스크롤 완료, 데이터 수집 시작
2 페이지 데이터 수집 완료 → 총 40건
3 페이지 스크롤 완료, 데이터 수집 시작
3 페이지 데이터 수집 완료 → 총 60건
{'page': 1, 'category': '블랙\n5성급\n호텔', 'name': '힐튼 경주', 'location': '경주시', 'rating': '9.4', 'reviews': '8,531명 평가', 'price': '158,400'}
{'page': 1, 'category': '블랙\n5성급\n호텔', 'name': '★당일특가★ 라한셀렉트 경주', 'location': '경주시', 'rating': '9.4', 'reviews': '2,437명 평가', 'price': '215,000'}
{'page': 1, 'category': '펜션', 'name': '경주 무드 펜션', 'location': '경주시', 'rating': '9.4', 'reviews': '127명 평가', 'price': '85,440'}
{'page': 1, 'category': '비즈니스\n호텔', 'name': '베스트웨스턴플러스 경주', 'location': '경주시', 'rating': '9.2', 'reviews': '1,875명 평가', 'price': '95,000'}
{'page': 1, 'category': '모텔', 'name': '경주 H Avenue 황리단길점', 'location': None, 'rating': '9.6', 'reviews': '675명 평가', 'price': None}
{'page': 1, 'category': '관광\n호텔', 'name': '경주시티호텔', 'location': '경주시', 'rating': '9.8', 'reviews': '90명 평가', 'price': '259,000'}
{'page': 1, 'category': '모텔', 'name': '경주 호텔 시리즈

In [24]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017/")
db = client["yeogi"]
collection = db["accommodations"]

# rating 내림차순, reviews 내림차순으로 정렬 후 10개
docs = collection.find({}, {"_id":0, "name":1, "rating":1, "reviews":1}) \
                 .sort([("rating", -1), ("reviews", -1)]) \
                 .limit(10)

for d in docs:
    print(d)

{'name': '경주 플럭스', 'rating': '9.9', 'reviews': '87명 평가'}
{'name': '경주 나의봄풀빌라', 'rating': '9.9', 'reviews': '447명 평가'}
{'name': '경주 펜타스케이프 풀빌라', 'rating': '9.9', 'reviews': '376명 평가'}
{'name': '경주 불국사한옥코코스테이', 'rating': '9.9', 'reviews': '10명 평가'}
{'name': '경주시티호텔', 'rating': '9.8', 'reviews': '90명 평가'}
{'name': '경주 호텔 시리즈', 'rating': '9.8', 'reviews': '878명 평가'}
{'name': '경주 브라운도트-경주보문점', 'rating': '9.8', 'reviews': '548명 평가'}
{'name': '경주 그리듯호텔', 'rating': '9.8', 'reviews': '236명 평가'}
{'name': '경주 보문 힐스테이 펜션', 'rating': '9.8', 'reviews': '234명 평가'}
{'name': '경주 데메테르풀빌라', 'rating': '9.8', 'reviews': '20명 평가'}
